In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Path
NOTEBOOK_DIR = Path().resolve()

if NOTEBOOK_DIR.name == "src":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

DATA_DIR = PROJECT_ROOT / "data" / "regime_change"
OUT_DIR  = PROJECT_ROOT / "results" / "regime_change"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR    :", DATA_DIR)
print("OUT_DIR     :", OUT_DIR)

## Data Smoothing

In [ ]:
def smooth(values, k):
    # I am smoothing the series using a simple moving average with radius k.
    # becaus since some files are very noisy, smoothing removes noise and makes the warm-up trend easier to detect.
    n = len(values)
    out = []
    for i in range(n):
        start = max(0, i - k) # LHS of the window.
        end   = min(n, i + k) # RHS of the window.
        total = 0.0
        count = 0
        for j in range(start, end):
            total += values[j] # I am summing values in the window.
            count += 1
        out.append(total / count) # storing the average for position i.
    return out

## Mean Range Calculation

In [ ]:
def mean_range(values, start, end):
    # Just computing the mean over a manual range [start, end).
    total = 0.0
    count = 0
    for i in range(start, end):
        total += values[i]
        count += 1
    if count == 0:
        return None
    return total / count

## Standard Deviation For Data Variation Around the Mean

In [ ]:
def std_all(values):

# I am computing the standard deviation in two phases: 1) mean 2) variance
    n = len(values)
    if n == 0:
        return 0.0

    # Mean
    total = 0.0
    for v in values:
        total += v
    mu = total / n

    # Variance
    var_sum = 0.0
    for v in values:
        diff = v - mu
        var_sum += diff * diff

    var = var_sum / n
    return var ** 0.5

## Dataset Loader

In [ ]:
def load_csv(filename):
    
    path = DATA_DIR / filename
    df = pd.read_csv(path, header=None)

    values = []
    first = True
    for v in df.iloc[:, 0]:
        if first:
            first = False
            continue  # I am skipping the header
        values.append(float(v))
    return values

## Change Point Detection Algorithm

In [ ]:
# Change Point Detector
# The idea over here is to detect a warm-up cutoff by searching for the largest drop in mean timing.
def detect_warmup(x, smooth_radius=20, window_size=200, abs_min_drop=0.25, rel_min_drop=0.4,
    max_pos_frac=0.8,    # warm-up must happen in first 80% of the run.
    global_min_drop=0.5  # early vs late mean must differ by at least this much
):
    
    n = len(x)

    # I reject the series if it is too short for my window sizes.
    if n < 2 * window_size + 10:
        return None

    # The smooth function will prevent false drops due to noise.
    xs = smooth(x, smooth_radius)

    # I compute the overall noise level to judge how big the drop is relative to noise.
    sigma = std_all(xs)
    if sigma == 0:
        return None

    # Now I am searching for the best split position this is obviously going to be the point with highest mean drop.
    best_t = None
    best_drop = 0.0

    # I restrict search to the first max_pos_frac * n positions.
    start_t = window_size
    end_t = min(n - window_size, int(max_pos_frac * n))

    for t in range(start_t, end_t):

        # Mean before the candidate cutoff
        m_before = mean_range(xs, t - window_size, t)
        # Mean after the candidate cutoff
        m_after  = mean_range(xs, t, t + window_size)

        if m_before is None or m_after is None:
            continue

        # I compute the drop. Positive drop means timing decreases basically my warm-up behavior.
        drop = m_before - m_after

        if drop > best_drop:
            best_drop = drop
            best_t = t

    if best_t is None:
        return None

    # Now, I am setting up the check to see if the drop is big enough. Had to adjust the hyperparameters quite a lot.
    if best_drop < abs_min_drop:
        return None
    if best_drop < rel_min_drop * sigma:
        return None

    # If the early and late phases look similar, then there is 0 warm-up.
    early_end   = max(window_size, int(0.2 * n)) # first 20%
    late_start  = min(n - window_size, int(0.6 * n))
    early_mean  = mean_range(xs, 0, early_end)
    late_mean   = mean_range(xs, late_start, n)

    if early_mean is None or late_mean is None:
        return None

    if abs(early_mean - late_mean) < global_min_drop:
        # This means the dataset is basically flat with no warm-up.
        return None

    return best_t

## Cut-Off Verification

In [ ]:
# The logic of this function is very important as it will tell me the mean before/after the cutoff. This way I think we can get some assurance on our cutoff
def cutoffVerification(x, t, window=300):
    n = len(x)
    xs = smooth(x, 20)
    before = mean_range(xs, max(0, t - window), t)
    after  = mean_range(xs, t, min(n, t + window))
    print(f"Cut {t}: mean_before={before:.3f}, mean_after={after:.3f}, drop={before-after:.3f}")

In [ ]:
# Plots
def timeSeriesPlot(x, cutoff, filename):
    plt.figure(figsize=(13, 5))
    plt.plot(x, label="Timing values")

    if cutoff is not None:
        plt.axvline(cutoff, color="red", linestyle="--",
                    label=f"Warm-up cutoff = {cutoff}")

    plt.title("Regime Change Detection: " + filename)
    plt.xlabel("Index")
    plt.ylabel("Timing")
    plt.legend()

    out_path = OUT_DIR / filename.replace(".csv", ".png")
    plt.savefig(out_path, dpi=300)
    plt.close()
    print("Saved plot:", out_path)

## Initial Analysis Using 1 Dataset

In [ ]:
# Analysis

# Just at first checking for 1 CSV file.
def analyze(filename):
    print("\n====", filename, "====")
    x = load_csv(filename)

    cutoff = detect_warmup(x, smooth_radius=20, window_size=200)


    if cutoff is None:
        print("NO warm-up detected.")
    else:
        print(f"Warmup cutoff at index {cutoff}")
        cutoffVerification(x, cutoff)

    timeSeriesPlot(x, cutoff, filename)

## Main

In [ ]:
if __name__ == "__main__":
    
    files = []

    for filename in os.listdir(DATA_DIR):
        if filename.endswith(".csv"):
            files.append(filename)

    if not files:
        print("No CSV files in", DATA_DIR)
    else:
        for f in files:
            analyze(f)

    print("\nAll done.")